# Filtering Pipelines

## What is this notebook doing?
This notebook evaluates four matching pipeline strategies, each computing comparisons independently to produce accurate timing measurements.

## Pipelines
1. **Visual only voting**: All 3 visual methods (pHash, dHash, HSV) vote — a pair is a match if 2 of 3 pass their threshold.
2. **Soft voting (all 4 methods)**: All 4 methods vote — a pair is a match if 2 of 4 pass.
3. **Hard filter — Audio first**: Audio gates the candidate pool. Pairs that fail the audio threshold are eliminated. Remaining pairs go to a 2-of-3 visual vote.
4. **Hard filter — pHash first**: pHash gates the candidate pool. Pairs that fail the pHash threshold are eliminated. Remaining pairs go to a 2-of-3 vote across audio, dHash, and HSV.

For hard filter pipelines, a pair must pass both the gate and the voting stage to be classified as a match. The gate reduces the candidate pool — only candidates that pass the gate are passed to the voting stage.

## Metrics
Each pipeline is evaluated on precision, recall, F1, total compute time, and number of pairs eliminated by the gate (hard filter pipelines only).

In [ ]:
import os
import re
import json
import time
import itertools
import numpy as np
import imagehash
import matplotlib.pyplot as plt
from scipy.spatial.distance import cosine as cosine_dist

# JSON paths
VISUAL_JSON = "visual_fingerprints.json"
AUDIO_JSON  = "audio_fingerprints.json"

# Configurable thresholds — copy final values from notebook 04
THRESHOLDS = {
    "phash":            {"min_diff": None, "mean_top_k": None},
    "dhash":            {"min_diff": None, "mean_top_k": None},
    "color_histogram":  {"min_diff": None, "mean_top_k": None},
    "chromaprint":      {"min_diff": None, "mean_top_k": None}
}

# Score type to use per method for voting and gating
# Set to "min_diff" or "mean_top_k" based on notebook 04 conclusions
SCORE_TYPE = {
    "phash":           "min_diff",
    "dhash":           "min_diff",
    "color_histogram": "min_diff",
    "chromaprint":     "min_diff"
}

TOP_K = 5
PREFIX_PATTERN = re.compile(r'^(.+)_[^_]+$')

In [ ]:
# --- Load fingerprints and build ground truth ---

with open(VISUAL_JSON, "r") as f:
    visual_data = json.load(f)
with open(AUDIO_JSON, "r") as f:
    audio_data = json.load(f)

video_names = list(visual_data.keys())
all_pairs = list(itertools.combinations(video_names, 2))

def extract_prefix(filename):
    name = os.path.splitext(filename)[0]
    match = PREFIX_PATTERN.match(name)
    return match.group(1) if match else name

prefix_to_videos = {}
for name in video_names:
    prefix = extract_prefix(name)
    prefix_to_videos.setdefault(prefix, []).append(name)

positive_pairs = set()
for videos in prefix_to_videos.values():
    for a, b in itertools.combinations(videos, 2):
        positive_pairs.add(frozenset([a, b]))

print(f"Videos: {len(video_names)} | Pairs: {len(all_pairs)} | Positive: {len(positive_pairs)} | Negative: {len(all_pairs) - len(positive_pairs)}")

In [ ]:
# --- Pairwise scoring functions ---

def get_visual_chunks(video_name, method):
    chunks = visual_data[video_name]["chunks"]
    result = []
    for chunk in chunks:
        frames = chunk["frames"]
        if not frames:
            continue
        if method in ("phash", "dhash"):
            result.append(imagehash.hex_to_hash(frames[0][method]))
        elif method == "color_histogram":
            vecs = np.array([f["color_histogram"] for f in frames])
            result.append(np.mean(vecs, axis=0))
    return result

def get_audio_chunks(video_name):
    return [
        np.array(chunk["fingerprint"], dtype=np.int32)
        for chunk in audio_data[video_name]["chunks"]
    ]

def visual_distance(c1, c2, method):
    if method in ("phash", "dhash"):
        return abs(c1 - c2)
    return cosine_dist(c1, c2)

def audio_distance(fp1, fp2):
    min_len = min(len(fp1), len(fp2))
    if min_len == 0:
        return 1.0
    xor = np.bitwise_xor(fp1[:min_len], fp2[:min_len])
    return sum(bin(x).count('1') for x in xor) / (min_len * 32)

def score_pair(chunks_a, chunks_b, method):
    """Compute min_diff and mean_top_k for a pair given preloaded chunks."""
    if method == "chromaprint":
        dists = [audio_distance(c1, c2) for c1 in chunks_a for c2 in chunks_b]
    else:
        dists = [visual_distance(c1, c2, method) for c1 in chunks_a for c2 in chunks_b]

    if not dists:
        return None, None

    dists_sorted = sorted(dists)
    return dists_sorted[0], float(np.mean(dists_sorted[:TOP_K]))

def passes_threshold(min_diff, mean_top_k, method):
    """Returns True if the pair passes the configured threshold for the method."""
    score_type = SCORE_TYPE[method]
    score = min_diff if score_type == "min_diff" else mean_top_k
    threshold = THRESHOLDS[method][score_type]
    if score is None or threshold is None:
        return False
    return score <= threshold

def evaluate_results(predictions, positive_pairs, all_pairs):
    """Compute precision, recall, F1 from a set of predicted match pairs."""
    tp = len(predictions & positive_pairs)
    fp = len(predictions - positive_pairs)
    fn = len(positive_pairs - predictions)
    tn = len(set(frozenset(p) for p in all_pairs) - positive_pairs - predictions)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"tp": tp, "fp": fp, "tn": tn, "fn": fn,
            "precision": precision, "recall": recall, "f1": f1}

print("Scoring functions ready.")

In [ ]:
# --- Pipeline 1: Visual only voting (2 of 3 visual methods must pass) ---

VISUAL_METHODS = ["phash", "dhash", "color_histogram"]

p1_predictions = set()
p1_start = time.time()

# Pre-load all visual chunks
p1_chunks = {m: {v: get_visual_chunks(v, m) for v in video_names} for m in VISUAL_METHODS}

for a, b in all_pairs:
    votes = 0
    for method in VISUAL_METHODS:
        min_diff, mean_top_k = score_pair(p1_chunks[method][a], p1_chunks[method][b], method)
        if passes_threshold(min_diff, mean_top_k, method):
            votes += 1
    if votes >= 2:
        p1_predictions.add(frozenset([a, b]))

p1_time = round(time.time() - p1_start, 3)
p1_metrics = evaluate_results(p1_predictions, positive_pairs, all_pairs)
print(f"Pipeline 1 (Visual voting) — Time: {p1_time}s | Matches found: {len(p1_predictions)}")
print(f"  Precision: {p1_metrics['precision']:.4f} | Recall: {p1_metrics['recall']:.4f} | F1: {p1_metrics['f1']:.4f}")

In [ ]:
# --- Pipeline 2: Soft voting (2 of 4 methods must pass) ---

ALL_METHODS = VISUAL_METHODS + ["chromaprint"]

p2_predictions = set()
p2_start = time.time()

p2_visual_chunks = {m: {v: get_visual_chunks(v, m) for v in video_names} for m in VISUAL_METHODS}
p2_audio_chunks  = {v: get_audio_chunks(v) for v in video_names}

for a, b in all_pairs:
    votes = 0
    for method in VISUAL_METHODS:
        min_diff, mean_top_k = score_pair(p2_visual_chunks[method][a], p2_visual_chunks[method][b], method)
        if passes_threshold(min_diff, mean_top_k, method):
            votes += 1
    min_diff, mean_top_k = score_pair(p2_audio_chunks[a], p2_audio_chunks[b], "chromaprint")
    if passes_threshold(min_diff, mean_top_k, "chromaprint"):
        votes += 1
    if votes >= 2:
        p2_predictions.add(frozenset([a, b]))

p2_time = round(time.time() - p2_start, 3)
p2_metrics = evaluate_results(p2_predictions, positive_pairs, all_pairs)
print(f"Pipeline 2 (Soft voting 4 methods) — Time: {p2_time}s | Matches found: {len(p2_predictions)}")
print(f"  Precision: {p2_metrics['precision']:.4f} | Recall: {p2_metrics['recall']:.4f} | F1: {p2_metrics['f1']:.4f}")

In [ ]:
# --- Pipeline 3: Hard filter — Audio first, then 2 of 3 visual vote ---

p3_predictions = set()
p3_eliminated = 0
p3_start = time.time()

p3_audio_chunks  = {v: get_audio_chunks(v) for v in video_names}
p3_visual_chunks = {m: {v: get_visual_chunks(v, m) for v in video_names} for m in VISUAL_METHODS}

for a, b in all_pairs:
    # Gate: audio
    min_diff, mean_top_k = score_pair(p3_audio_chunks[a], p3_audio_chunks[b], "chromaprint")
    if not passes_threshold(min_diff, mean_top_k, "chromaprint"):
        p3_eliminated += 1
        continue

    # Voting: 2 of 3 visual
    votes = 0
    for method in VISUAL_METHODS:
        min_diff, mean_top_k = score_pair(p3_visual_chunks[method][a], p3_visual_chunks[method][b], method)
        if passes_threshold(min_diff, mean_top_k, method):
            votes += 1
    if votes >= 2:
        p3_predictions.add(frozenset([a, b]))

p3_time = round(time.time() - p3_start, 3)
p3_metrics = evaluate_results(p3_predictions, positive_pairs, all_pairs)
print(f"Pipeline 3 (Audio gate → Visual voting) — Time: {p3_time}s | Matches found: {len(p3_predictions)}")
print(f"  Pairs eliminated by gate: {p3_eliminated} / {len(all_pairs)}")
print(f"  Precision: {p3_metrics['precision']:.4f} | Recall: {p3_metrics['recall']:.4f} | F1: {p3_metrics['f1']:.4f}")

In [ ]:
# --- Pipeline 4: Hard filter — pHash first, then 2 of 3 vote (audio + dHash + HSV) ---

P4_VOTE_METHODS = ["chromaprint", "dhash", "color_histogram"]

p4_predictions = set()
p4_eliminated = 0
p4_start = time.time()

p4_phash_chunks = {v: get_visual_chunks(v, "phash") for v in video_names}
p4_audio_chunks = {v: get_audio_chunks(v) for v in video_names}
p4_visual_chunks = {m: {v: get_visual_chunks(v, m) for v in video_names} for m in ["dhash", "color_histogram"]}

for a, b in all_pairs:
    # Gate: pHash
    min_diff, mean_top_k = score_pair(p4_phash_chunks[a], p4_phash_chunks[b], "phash")
    if not passes_threshold(min_diff, mean_top_k, "phash"):
        p4_eliminated += 1
        continue

    # Voting: 2 of 3 (audio + dHash + HSV)
    votes = 0
    min_diff, mean_top_k = score_pair(p4_audio_chunks[a], p4_audio_chunks[b], "chromaprint")
    if passes_threshold(min_diff, mean_top_k, "chromaprint"):
        votes += 1
    for method in ["dhash", "color_histogram"]:
        min_diff, mean_top_k = score_pair(p4_visual_chunks[method][a], p4_visual_chunks[method][b], method)
        if passes_threshold(min_diff, mean_top_k, method):
            votes += 1
    if votes >= 2:
        p4_predictions.add(frozenset([a, b]))

p4_time = round(time.time() - p4_start, 3)
p4_metrics = evaluate_results(p4_predictions, positive_pairs, all_pairs)
print(f"Pipeline 4 (pHash gate → Audio+dHash+HSV voting) — Time: {p4_time}s | Matches found: {len(p4_predictions)}")
print(f"  Pairs eliminated by gate: {p4_eliminated} / {len(all_pairs)}")
print(f"  Precision: {p4_metrics['precision']:.4f} | Recall: {p4_metrics['recall']:.4f} | F1: {p4_metrics['f1']:.4f}")

In [ ]:
# --- Pipeline comparison summary ---

pipelines = [
    {"name": "Visual voting (3 methods)",         "metrics": p1_metrics, "time": p1_time, "eliminated": None},
    {"name": "Soft voting (4 methods)",            "metrics": p2_metrics, "time": p2_time, "eliminated": None},
    {"name": "Hard filter: Audio → Visual vote",   "metrics": p3_metrics, "time": p3_time, "eliminated": p3_eliminated},
    {"name": "Hard filter: pHash → Audio+2 vote",  "metrics": p4_metrics, "time": p4_time, "eliminated": p4_eliminated},
]

print(f"{'Pipeline':<40} {'Time (s)':>9} {'Eliminated':>11} {'TP':>4} {'FP':>4} {'FN':>4} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("-" * 106)
for p in pipelines:
    m = p["metrics"]
    elim = str(p["eliminated"]) if p["eliminated"] is not None else "N/A"
    print(
        f"{p['name']:<40} {p['time']:>9.3f} {elim:>11} "
        f"{m['tp']:>4} {m['fp']:>4} {m['fn']:>4} "
        f"{m['precision']:>10.4f} {m['recall']:>8.4f} {m['f1']:>8.4f}"
    )

# Bar charts
labels = [p["name"] for p in pipelines]
x = np.arange(len(labels))
width = 0.22

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Accuracy metrics
axes[0].bar(x - width, [p["metrics"]["precision"] for p in pipelines], width, label="Precision", color="steelblue")
axes[0].bar(x,         [p["metrics"]["recall"]    for p in pipelines], width, label="Recall",    color="salmon")
axes[0].bar(x + width, [p["metrics"]["f1"]        for p in pipelines], width, label="F1",        color="mediumseagreen")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=15, ha="right", fontsize=9)
axes[0].set_ylim(0, 1.1)
axes[0].set_title("Accuracy Metrics by Pipeline")
axes[0].legend()

# Compute time
axes[1].bar(labels, [p["time"] for p in pipelines], color="mediumpurple", alpha=0.8)
axes[1].set_xticklabels(labels, rotation=15, ha="right", fontsize=9)
axes[1].set_title("Compute Time by Pipeline (seconds)")
axes[1].set_ylabel("Seconds")

plt.suptitle("Pipeline Comparison Summary", fontsize=13)
plt.tight_layout()
plt.show()

## Conclusion

**Pipeline chosen:** 

**Reasoning:** 